# 01 — Delta Lake on MinIO

Create a Delta table on `s3a://spark-warehouse/delta/people/`, mutate it, then time-travel back.

In [1]:
from spark_session import get_spark
from pyspark.sql import functions as F

spark = get_spark("01-delta")
TABLE = "s3a://spark-warehouse/delta/people"

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 13:14:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/11 13:14:42 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/06/11 13:14:43 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to events/app-20260611131442-0002.inprogress. This is unsupported


In [3]:
people = spark.createDataFrame(
    [(1, "Ada", 36), (2, "Linus", 54), (3, "Grace", 85), (4, "Dennis", 70)],
    ["id", "name", "age"],
)
people.show()
people.write.format("delta").mode("overwrite").save(TABLE)
spark.read.format("delta").load(TABLE).show()

+---+------+---+
| id|  name|age|
+---+------+---+
|  1|   Ada| 36|
|  2| Linus| 54|
|  3| Grace| 85|
|  4|Dennis| 70|
+---+------+---+



+---+------+---+
| id|  name|age|
+---+------+---+
|  3| Grace| 85|
|  4|Dennis| 70|
|  1|   Ada| 36|
|  2| Linus| 54|
+---+------+---+



In [4]:
# Register a SQL view so we can use UPDATE / DELETE.
spark.sql(f"CREATE OR REPLACE TEMP VIEW people AS SELECT * FROM delta.`{TABLE}`")
spark.sql(f"UPDATE delta.`{TABLE}` SET age = age + 1 WHERE name = 'Ada'")
spark.sql(f"DELETE FROM delta.`{TABLE}` WHERE name = 'Dennis'")
spark.read.format("delta").load(TABLE).orderBy("id").show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|  Ada| 37|
|  2|Linus| 54|
|  3|Grace| 85|
+---+-----+---+



In [5]:
# Full history — should be at least 3 versions: write, update, delete.
spark.sql(f"DESCRIBE HISTORY delta.`{TABLE}`").select("version", "timestamp", "operation").show(truncate=False)

+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|39     |2026-06-11 13:16:23|DELETE   |
|38     |2026-06-11 13:16:21|UPDATE   |
|37     |2026-06-11 13:15:27|WRITE    |
|36     |2026-06-11 13:15:01|WRITE    |
|35     |2026-06-03 19:01:42|DELETE   |
|34     |2026-06-03 19:01:40|UPDATE   |
|33     |2026-06-03 19:01:35|WRITE    |
|32     |2026-06-03 18:58:27|DELETE   |
|31     |2026-06-03 18:58:25|UPDATE   |
|30     |2026-06-03 18:58:17|WRITE    |
|29     |2026-06-03 18:48:46|DELETE   |
|28     |2026-06-03 18:48:44|UPDATE   |
|27     |2026-06-03 18:48:39|WRITE    |
|26     |2026-06-03 18:44:41|DELETE   |
|25     |2026-06-03 18:44:39|UPDATE   |
|24     |2026-06-03 18:44:35|WRITE    |
|23     |2026-06-03 18:39:25|DELETE   |
|22     |2026-06-03 18:39:23|UPDATE   |
|21     |2026-06-03 18:39:18|WRITE    |
|20     |2026-06-03 18:33:46|DELETE   |
+-------+-------------------+---------+
only showing top 20 rows



In [6]:
# Time-travel: read version 0 (the original insert).
(spark.read.format("delta").option("versionAsOf", 0).load(TABLE).orderBy("id").show())

+---+------+---+
| id|  name|age|
+---+------+---+
|  1|   Ada| 36|
|  2| Linus| 54|
|  3| Grace| 85|
|  4|Dennis| 70|
+---+------+---+



Inspect the underlying objects in the MinIO console at http://localhost:9001 → bucket `spark-warehouse` → `delta/people/_delta_log/`.

In [7]:
# Release the SparkSession so cluster cores free up and the History Server can ingest this app.
spark.stop()